<a href="https://colab.research.google.com/github/AhmedCode110/AC-MOT/blob/main/notebooks/AC_MOT_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AC-MOT portable Colab runner
Use a Tesla T4 runtime for GPU modes. The notebook validates the Colab Secret `GITHUB_TOKEN`, verifies that it can read `AhmedCode110/AC-MOT`, pulls the latest private repository, installs dependencies, loads the active config, performs preflight checks, and runs the selected experiment.

In [ ]:
print('[1/6] Mounting Google Drive...', flush=True)
from google.colab import drive
drive.mount('/content/drive')
print('[OK] Google Drive mounted.', flush=True)

## Automatic private GitHub authentication
Create a Colab Secret named `GITHUB_TOKEN` once and enable Notebook access. The notebook first checks the token against GitHub and verifies access to this private repository. The token is never printed, written into the notebook, repository URL, or Git configuration.

In [ ]:
print('[2/6] Syncing latest AC-MOT code from GitHub...', flush=True)
from pathlib import Path
import json, os, sys, subprocess, tempfile, urllib.request, urllib.error
from google.colab import userdata

REPO_URL = 'https://github.com/AhmedCode110/AC-MOT.git'
REPO_API = 'https://api.github.com/repos/AhmedCode110/AC-MOT'
USER_API = 'https://api.github.com/user'
REPO = Path('/content/AC-MOT')

try:
    token = userdata.get('GITHUB_TOKEN')
except Exception as exc:
    raise RuntimeError('Colab Secret GITHUB_TOKEN is missing or Notebook access is disabled. Add it in the Secrets panel and enable Notebook access.') from exc
if not token:
    raise RuntimeError('Colab Secret GITHUB_TOKEN is empty.')

def github_json(url):
    request = urllib.request.Request(url, headers={
        'Authorization': f'Bearer {token}',
        'Accept': 'application/vnd.github+json',
        'X-GitHub-Api-Version': '2022-11-28',
        'User-Agent': 'AC-MOT-Colab'
    })
    try:
        with urllib.request.urlopen(request, timeout=20) as response:
            return response.status, json.loads(response.read().decode('utf-8'))
    except urllib.error.HTTPError as exc:
        body = exc.read().decode('utf-8', errors='replace')
        try: detail = json.loads(body).get('message', body)
        except Exception: detail = body
        return exc.code, {'message': detail}

print('[GITHUB] Validating saved token...', flush=True)
user_status, user_data = github_json(USER_API)
if user_status != 200:
    raise RuntimeError(f'GITHUB_TOKEN is not valid for GitHub API authentication (HTTP {user_status}: {user_data.get("message")}). Replace the Colab Secret with a valid PAT.')
github_user = user_data.get('login')
print(f'[OK] Token valid | GitHub user={github_user}', flush=True)

print('[GITHUB] Verifying private repository access...', flush=True)
repo_status, repo_data = github_json(REPO_API)
if repo_status != 200:
    raise RuntimeError(f'Token is valid but cannot read AhmedCode110/AC-MOT (HTTP {repo_status}: {repo_data.get("message")}). For a fine-grained PAT, select repository AC-MOT and grant Contents: Read-only.')
print(f'[OK] Repository access confirmed | {repo_data.get("full_name")}', flush=True)

with tempfile.TemporaryDirectory() as tmp:
    env = os.environ.copy()
    env['GIT_TERMINAL_PROMPT'] = '0'
    env['ACMOT_GH_TOKEN'] = token
    env['ACMOT_GH_USER'] = github_user
    helper = Path(tmp) / 'askpass'
    helper.write_text(
        '#!/usr/bin/env python3\n'
        'import os, sys\n'
        'prompt = sys.argv[1] if len(sys.argv) > 1 else ""\n'
        'print(os.environ["ACMOT_GH_USER"] if "Username" in prompt else os.environ["ACMOT_GH_TOKEN"])\n'
    )
    helper.chmod(0o700)
    env['GIT_ASKPASS'] = str(helper)
    command = ['git', '-c', 'credential.helper=']
    if (REPO / '.git').is_dir():
        print('[GITHUB] Repository exists; pulling origin/main...', flush=True)
        result = subprocess.run(command + ['-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], env=env, text=True, capture_output=True)
    else:
        print('[GITHUB] Repository not present; cloning main...', flush=True)
        result = subprocess.run(command + ['clone', '--branch', 'main', REPO_URL, str(REPO)], env=env, text=True, capture_output=True)
    if result.stdout.strip(): print(result.stdout, flush=True)
    if result.stderr.strip(): print(result.stderr, flush=True)
    result.check_returncode()

commit = subprocess.check_output(['git','-C',str(REPO),'rev-parse','--short','HEAD'], text=True).strip()
print(f'[OK] Repository ready at {REPO} | commit={commit}', flush=True)

In [ ]:
print('[3/6] Installing pinned dependencies and TrackEval...', flush=True)
subprocess.run([sys.executable,'-m','pip','install','-r',str(REPO/'requirements.txt')],check=True)
subprocess.run([sys.executable,str(REPO/'scripts/setup_trackeval.py'),'/content/TrackEval'],check=True)
print('[OK] Dependencies ready.', flush=True)

## Central configuration
The active configuration is selected by `configs/active_config.txt`. GitHub is the source of truth; Colab is only the runner.

In [ ]:
print('[4/6] Loading active configuration and checking Drive paths...', flush=True)
import json, uuid, shutil
CONFIG_NAME = (REPO / 'configs' / 'active_config.txt').read_text().strip()
print('Active config:', CONFIG_NAME, flush=True)
CFG=json.loads((REPO/'configs'/CONFIG_NAME).read_text())
COPY_DATASET_TO_LOCAL=False
DATASET_ON_DRIVE=Path(CFG['dataset'])
assert (DATASET_ON_DRIVE/'annotations').is_dir(), f'Check Drive access/path: {DATASET_ON_DRIVE}'
if COPY_DATASET_TO_LOCAL:
    local=Path('/content')/('acmot_dataset_'+uuid.uuid4().hex[:8])/DATASET_ON_DRIVE.name
    shutil.copytree(DATASET_ON_DRIVE,local)
    CFG['dataset']=str(local)
RESULTS=Path(CFG['output_root'])
RESULTS.mkdir(parents=True,exist_ok=True)
probe=RESULTS/('.write_probe_'+uuid.uuid4().hex)
probe.write_text('probe');probe.unlink()
CONFIG_PATH=Path('/content')/('acmot_config_'+uuid.uuid4().hex+'.json')
CONFIG_PATH.write_text(json.dumps(CFG,indent=2))
print(f'[OK] Config ready: {CONFIG_PATH}', flush=True)

In [ ]:
print('[5/6] Running environment preflight...', flush=True)
subprocess.run([sys.executable,str(REPO/'scripts/run.py'),'--config',str(CONFIG_PATH),'--check'],check=True)
print('[OK] Preflight passed.', flush=True)

In [ ]:
print('[6/6] Starting active AC-MOT run...', flush=True)
subprocess.run([sys.executable,str(REPO/'scripts/run.py'),'--config',str(CONFIG_PATH)],check=True)
print('[OK] Run completed.', flush=True)
print('Results root on Drive:', CFG['output_root'], flush=True)

## Daily workflow
Edit and push code from VS Code, open this notebook, then choose Run all. The notebook validates `GITHUB_TOKEN`, verifies private-repo access, pulls the latest `main`, installs the pinned environment, loads the active config, checks the runtime, and runs the experiment.